# Task 4: Data Storytelling & Statistical Validation

## 📌 Executive Overview
This Jupyter Notebook contains the statistical analysis and validation workflow for **Task 4**. We analyze an A/B test comparing two website layouts:
- **Group A (Control / Old Layout):** Legacy checkout and product interface.
- **Group B (Variant / New Layout):** Streamlined layout designed to boost user conversions.

---

## 🎯 Objectives
1. Perform descriptive statistics on conversion rate metrics across both groups.
2. Formulate and test statistical hypotheses using Welch's Two-Sample Independent $T$-Test.
3. Compute $95\%$ Confidence Intervals for the difference in conversion means.
4. Export key summary metrics for stakeholder presentation and reporting.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Styling configuration
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("Environment successfully configured!")

## 1. Dataset Generation & Exploratory Inspection
We simulate $N = 100$ cohort sample batches per group:
- **Group A Mean:** $9.80\%$ ($\sigma = 2.0\%$)
- **Group B Mean:** $12.05\%$ ($\sigma = 2.0\%$)

In [ ]:
# Simulate conversion rate cohorts
group_a = np.random.normal(loc=0.0980, scale=0.0200, size=100)
group_b = np.random.normal(loc=0.1205, scale=0.0200, size=100)

# Create Pandas DataFrame
df = pd.DataFrame({
    'Group_A_Old_Layout': group_a,
    'Group_B_New_Layout': group_b
})

# Display first 5 rows
df.head()

## 2. Descriptive Statistics Summary

In [ ]:
summary_stats = df.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
summary_stats['mean_percentage'] = summary_stats['mean'] * 100
summary_stats

## 3. Hypothesis Testing Framework

### Hypotheses Formulation
- **Null Hypothesis ($H_0$):** $\mu_B - \mu_A = 0$ *(The new layout has no effect on conversion rates)*
- **Alternative Hypothesis ($H_1$):** $\mu_B - \mu_A > 0$ *(The new layout significantly increases conversion rates)*

### Methodology
- **Significance Threshold ($\alpha$):** $0.05$ ($95\%$ Confidence Level)
- **Statistical Test:** Welch's Two-Sample Independent $T$-Test (`scipy.stats.ttest_ind` with `equal_var=False`)

In [ ]:
mean_a = df['Group_A_Old_Layout'].mean()
mean_b = df['Group_B_New_Layout'].mean()
std_a = df['Group_A_Old_Layout'].std()
std_b = df['Group_B_New_Layout'].std()
n_a, n_b = len(df['Group_A_Old_Layout']), len(df['Group_B_New_Layout'])

# Calculate Welch's T-Test
t_stat, p_value = stats.ttest_ind(df['Group_A_Old_Layout'], df['Group_B_New_Layout'], equal_var=False)

# Calculate 95% Confidence Interval for Difference in Means
diff = mean_b - mean_a
se_diff = np.sqrt((std_a**2 / n_a) + (std_b**2 / n_b))
degrees_of_freedom = n_a + n_b - 2
ci_factor = stats.t.ppf(0.975, df=degrees_of_freedom)

margin_of_error = ci_factor * se_diff
ci_lower = diff - margin_of_error
ci_upper = diff + margin_of_error

print("=== STATISTICAL HYPOTHESIS TEST RESULTS ===")
print(f"Group A Mean Conversion  : {mean_a*100:.2f}%")
print(f"Group B Mean Conversion  : {mean_b*100:.2f}%")
print(f"Absolute Difference      : +{diff*100:.2f}% percentage points")
print(f"Relative Lift            : +{((mean_b - mean_a)/mean_a)*100:.2f}%")
print(f"T-Statistic              : {t_stat:.4f}")
print(f"P-Value                  : {p_value:.6e}")
print(f"95% Confidence Interval  : [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
print("-" * 43)

if p_value < 0.05:
    print("DECISION: REJECT THE NULL HYPOTHESIS (H0)")
    print("CONCLUSION: Statistically significant improvement detected in Group B.")
else:
    print("DECISION: FAIL TO REJECT THE NULL HYPOTHESIS (H0)")

## 4. Visualizing Results & Exporting Outputs

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(df['Group_A_Old_Layout'], fill=True, color='#ef4444', label='Group A (Old Layout)', alpha=0.4)
sns.kdeplot(df['Group_B_New_Layout'], fill=True, color='#34d399', label='Group B (New Layout)', alpha=0.4)

plt.axvline(mean_a, color='#ef4444', linestyle='--', linewidth=2, label=f'Mean A ({mean_a*100:.2f}%)')
plt.axvline(mean_b, color='#34d399', linestyle='--', linewidth=2, label=f'Mean B ({mean_b*100:.2f}%)')

plt.title('Conversion Rate Distribution Comparison (A/B Testing)', fontsize=14, fontweight='bold')
plt.xlabel('Conversion Rate')
plt.ylabel('Density')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# Export Summary CSV
summary_df = pd.DataFrame([{
    'Group_A_Mean': f"{mean_a*100:.2f}%",
    'Group_B_Mean': f"{mean_b*100:.2f}%",
    'Relative_Lift': f"{((mean_b - mean_a)/mean_a)*100:.2f}%",
    'T_Statistic': round(t_stat, 4),
    'P_Value': f"{p_value:.6e}",
    'CI_95_Lower': f"{ci_lower*100:.2f}%",
    'CI_95_Upper': f"{ci_upper*100:.2f}%",
    'Decision': 'Reject Null Hypothesis' if p_value < 0.05 else 'Fail to Reject'
}])
summary_df.to_csv('hypothesis_testing_results.csv', index=False)
print("Saved summary metrics to hypothesis_testing_results.csv")